# PropertyLens — fresh clone setup

Run this notebook once after `git clone` to:

1. **Create folders** under `data/` where models, feature tables, and amenities are expected.
2. **Prepare environment files** (`.env` for Hugging Face + pointers to `backend/.env`).
3. **Download large artefacts** from Hugging Face — model bundle, feature-layer CSVs, and **amenity POI CSVs** (requires an HF token).
4. **Initialize the SQLite database** used for auth, prediction history, and wishlist.
5. **Verify** that the backend can find bundles, feature tables, and map amenities.

**Prerequisites:** Python 3.11+ recommended (matches sklearn/XGBoost pickles). Use a venv at the repo root or `backend/.venv` and install `pip install -r backend/requirements.txt` plus `jupyter` / `ipykernel` if needed.

> Large files are **gitignored** — they are not in the repository. Download them with this notebook (from Hugging Face) or copy from a teammate. To **regenerate** amenities from government APIs instead, use `notebooks/00_download_amenity_data.ipynb`.

## 1. Repo root and directories

In [ ]:
from pathlib import Path

_cwd = Path.cwd().resolve()
REPO_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd

DATA = REPO_ROOT / "data"
ARTIFACTS = DATA / "artifacts"
FEATURE_OUT = DATA / "feature_data" / "02_feature_layer" / "training" / "outputs"
AMENITIES = DATA / "amenities"

for p in (DATA, ARTIFACTS, ARTIFACTS / "hybrid_xai", FEATURE_OUT, AMENITIES):
    p.mkdir(parents=True, exist_ok=True)

print("REPO_ROOT =", REPO_ROOT)
print("Created / ensured:", DATA, ARTIFACTS, FEATURE_OUT, AMENITIES, sep="\n  ")

## 2. Environment files

- **Repo root `.env`**: used by these notebooks for `HF_TOKEN` (Hugging Face).
- **`backend/.env`**: optional overrides for `DATABASE_URL`, `PROPERTYLENS_ARTIFACTS_DIR`, chat, etc. Copy from `backend/.env.example`.
- **`frontend/.env`**: optional `VITE_API_BASE_URL`. Copy from `frontend/.env.example`.

The cell below creates **root `.env` only if missing**, with a placeholder for `HF_TOKEN`.

In [ ]:
ROOT_ENV = REPO_ROOT / ".env"
if ROOT_ENV.exists():
    print("Exists (unchanged):", ROOT_ENV)
else:
    ROOT_ENV.write_text(
        "# Hugging Face — required for download cells below\n"
        "# Create a read token at https://huggingface.co/settings/tokens\n"
        "HF_TOKEN=\n",
        encoding="utf-8",
    )
    print("Created", ROOT_ENV, "— add HF_TOKEN=hf_... then re-run download cells.")

be = REPO_ROOT / "backend" / ".env.example"
bt = REPO_ROOT / "backend" / ".env"
if be.exists() and not bt.exists():
    print("Tip: copy backend env:  cp backend/.env.example backend/.env")
elif bt.exists():
    print("backend/.env present")

## 3. Download model artefacts (Hugging Face)

Pulls `artifacts/**` from [`PropertyLens/propertylens-models`](https://huggingface.co/PropertyLens/propertylens-models) into `data/` (so you get `data/artifacts/...`).

Requires `HF_TOKEN` in the repo-root `.env`.

In [ ]:
%pip install -q python-dotenv huggingface-hub

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import snapshot_download

load_dotenv(REPO_ROOT / ".env")
hf_token = os.getenv("HF_TOKEN", "").strip()
if not hf_token:
    raise ValueError("Set HF_TOKEN in repo-root .env (see previous section).")

LOCAL_DATA = REPO_ROOT / "data"
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

path = snapshot_download(
    repo_id="PropertyLens/propertylens-models",
    repo_type="model",
    token=hf_token,
    local_dir=str(LOCAL_DATA),
    allow_patterns=["artifacts/**"],
)
print("Models snapshot:", path)

## 4. Download feature-layer tables (Hugging Face)

Downloads `02_feature_layer/training/outputs/**` from the dataset repo into `data/feature_data/...` (used for address lookup and trends).

Adjust `REPO_ID` / `ALLOW` if your dataset path differs.

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from huggingface_hub import snapshot_download

load_dotenv(REPO_ROOT / ".env")
hf_token = os.getenv("HF_TOKEN", "").strip()
if not hf_token:
    raise ValueError("Set HF_TOKEN in repo-root .env")

out = REPO_ROOT / "data" / "feature_data"
out.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id="PropertyLens/Resealeflats",
    repo_type="dataset",
    token=hf_token,
    local_dir=str(out),
    allow_patterns=["02_feature_layer/training/outputs/**"],
)
print("Feature outputs under:", out)

## 5. Download amenity CSVs (Hugging Face)

Pulls **`amenities/**`** from the dataset [`PropertyLens/Resealeflats`](https://huggingface.co/datasets/PropertyLens/Resealeflats) into **`data/amenities/`** (MRT, hawker centres, schools, malls — used by the map / nearby POIs).

Uses the same `HF_TOKEN` as above. To publish updates to the Hub, see `notebooks/00_test_upload_amenities_to_hf.ipynb`. To build CSVs from scratch via data.gov.sg / OneMap, use `00_download_amenity_data.ipynb` instead.

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import snapshot_download

load_dotenv(REPO_ROOT / ".env")
hf_token = os.getenv("HF_TOKEN", "").strip()
if not hf_token:
    raise ValueError("Set HF_TOKEN in repo-root .env")

LOCAL_DATA = REPO_ROOT / "data"
LOCAL_DATA.mkdir(parents=True, exist_ok=True)

snapshot_download(
    repo_id="PropertyLens/Resealeflats",
    repo_type="dataset",
    token=hf_token,
    local_dir=str(LOCAL_DATA),
    allow_patterns=["amenities/**"],
)
print("Amenities under:", LOCAL_DATA / "amenities")

## 6. Initialize SQLite database

Creates tables for users, prediction history, and wishlist at **`data/propertylens.db`** (default). Override with `DATABASE_URL` in `backend/.env` if you use Postgres, etc.

In [ ]:
import sys

BACKEND = REPO_ROOT / "backend"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))

from db import init_db, REPO_ROOT as DB_REPO, DATABASE_URL

assert DB_REPO.resolve() == REPO_ROOT.resolve(), (DB_REPO, REPO_ROOT)
init_db()
print("init_db() OK")
print("DATABASE_URL =", DATABASE_URL)

## 7. Verification checklist

The FastAPI app loads **`data/artifacts/hybrid_cluster_bundle.joblib`** and **`data/artifacts/hybrid_xai/*`** by default. Feature tables: latest `hdb_feature_table_*.csv` under `data/feature_data/.../outputs/` unless `PROPERTYLENS_FEATURE_TABLE_CSV` is set. Amenity POIs: **`data/amenities/*.csv`** (from §5) for maps / nearby endpoints.

In [ ]:
from pathlib import Path

checks = [
    REPO_ROOT / "data" / "artifacts" / "hybrid_cluster_bundle.joblib",
    REPO_ROOT / "data" / "artifacts" / "hybrid_cluster_feature_columns.json",
    REPO_ROOT / "data" / "artifacts" / "hybrid_xai" / "lime_training_data.joblib",
    REPO_ROOT / "data" / "artifacts" / "hybrid_xai" / "cbr_training_data.parquet",
]
amenity_names = (
    "mrt_stations.csv",
    "hawker_centres.csv",
    "schools.csv",
    "malls.csv",
)
tables = sorted((REPO_ROOT / "data" / "feature_data").rglob("hdb_feature_table_*.csv"))

ok = True
for p in checks:
    ex = p.exists()
    ok = ok and ex
    print(("OK " if ex else "MISSING"), p.relative_to(REPO_ROOT))

if tables:
    print("Feature tables (sample):", tables[-1].relative_to(REPO_ROOT))
else:
    print("MISSING: no hdb_feature_table_*.csv under data/feature_data")
    ok = False

am_root = REPO_ROOT / "data" / "amenities"
for name in amenity_names:
    p = am_root / name
    ex = p.is_file()
    ok = ok and ex
    print(("OK " if ex else "MISSING"), p.relative_to(REPO_ROOT))

db = REPO_ROOT / "data" / "propertylens.db"
print(("OK " if db.exists() else "MISSING"), db.relative_to(REPO_ROOT))

print("\nAll critical artefacts:", "yes" if ok and db.exists() else "no — run download sections above")

## 8. Next steps (terminal)

```bash
# Backend (from repo root, venv active)
pip install -r backend/requirements.txt
cd backend && uvicorn main:app --reload --port 8000

# Frontend (separate terminal)
cd frontend && npm install && npm run dev
```

Health check: [http://127.0.0.1:8000/health](http://127.0.0.1:8000/health)

More detail: `backend/.env.example`, `backend/README.md`, `docs/views-developer.md`.